# 02 - LangGraph CRM Agent

本筆記本示範一個 **ReAct Agent**，透過反覆呼叫三個工具（`list_files`、`grep`、`read_file`）來回答關於 CRM 會議紀錄的問題。所有工具的存取範圍僅限於 `data/crm_notes/` 目錄。
> **注意**：需要在專案根目錄建立 `.env` 檔案（可複製 `.env.example` 後填入實際值）。

## 架構概覽

```
START → [agent] → route() → [tools]      → [agent] → ...
                          → [text_tools]  → [agent] → ...
                          → END
```

Agent 每次執行後由 `route()` 函式決定下一步：

- **`tools`**：模型輸出了結構化的 function_calls（標準 OpenAI 格式），交由 LangGraph 內建的 `ToolNode` 執行。
- **`text_tools`**：Gemma-4 fallback 路徑——模型可能以純文字輸出 `call:func_name{key:val}` 格式的工具呼叫。`text_tool_node` 以正規表達式解析此格式，執行工具，並將結果拼接為標準的 `ToolMessage` 後送回 agent 繼續推理。
- **`END`**：模型已完成推理，直接輸出最終答案。

In [1]:
import os
import re
import sys
import uuid
from pathlib import Path
from typing import Literal

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / "examples"
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, RemoveMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from utils.tools import FileTools

load_dotenv(PROJECT_ROOT / ".env")
print(f"Project root: {PROJECT_ROOT}")

Project root: /home/mistin/research/agentic-rag-lab


## 1 · Tools — 限定在 crm_notes/

`FileTools` 是一個以 `base_dir` 為根目錄的檔案存取工具集，所有路徑操作皆被限縮在指定目錄內，防止 agent 存取範圍外的檔案。

`StructuredTool.from_function()` 會自動從函式的 docstring 擷取工具說明，LLM 在選擇工具時會參考這段描述，因此撰寫清晰的 docstring 對 agent 的推理品質至關重要。

In [2]:
_crm = FileTools(PROJECT_ROOT / "data" / "crm_notes")
tools = [
    StructuredTool.from_function(_crm.list_files),
    StructuredTool.from_function(_crm.grep),
    StructuredTool.from_function(_crm.read_file),
]
_tools_by_name = {t.name: t for t in tools}
print("Registered tools:", [t.name for t in tools])

Registered tools: ['list_files', 'grep', 'read_file']


## 2 · LLM — OpenAI-compatible vLLM endpoint

透過 `ChatOpenAI` 連接本地 vLLM 服務，設定來自 `.env` 中的環境變數。`bind_tools()` 將工具描述（schema）附加到每次請求，讓模型知道可以呼叫哪些工具。

In [3]:
llm = ChatOpenAI(
    base_url=os.environ["VLLM_BASE_URL"],
    api_key=os.environ["VLLM_API_KEY"],
    model=os.environ["VLLM_MODEL"],
    temperature=0,
)
llm_with_tools = llm.bind_tools(tools)
print(f"Model: {os.environ['VLLM_MODEL']}")

Model: gemma-4-31B-it


In [5]:
SYSTEM_PROMPT = (
    "你是 CRM 資料分析助理，只能依據會議紀錄回答問題。\n"
    "可用工具：\n"
    "  - list_files(pattern='*.md')  列出所有會議紀錄檔案\n"
    "  - grep(pattern, max_results)  在所有檔案中搜尋關鍵字\n"
    "  - read_file(path)             讀取指定檔案的完整內容\n"
    "搜尋策略：先 list_files 確認檔案清單，再 grep 定位關鍵段落，再 read_file 閱讀全文。"
    "需要時可多次呼叫工具。\n"
    "回答時引用來源檔名，資料不足請如實說明。"
)
print(SYSTEM_PROMPT)

你是 CRM 資料分析助理，只能依據會議紀錄回答問題。
可用工具：
  - list_files(pattern='*.md')  列出所有會議紀錄檔案
  - grep(pattern, max_results)  在所有檔案中搜尋關鍵字
  - read_file(path)             讀取指定檔案的完整內容
搜尋策略：先 list_files 確認檔案清單，再 grep 定位關鍵段落，再 read_file 閱讀全文。需要時可多次呼叫工具。
回答時引用來源檔名，資料不足請如實說明。


## 3 · Gemma-4 Fallback Parser

Gemma-4 有時不會輸出標準的 OpenAI function_calls 格式，而是在純文字回覆中夾帶工具呼叫，格式如下：

```
call:list_files{pattern:*.md}
call:grep{pattern:晨星科技,max_results:5}
```

正規表達式 `_TEXT_TOOL_RE` 會偵測此格式。一旦命中，`text_tool_node` 會：
1. 解析函式名稱與參數
2. 執行對應工具
3. 用 `RemoveMessage` 移除原本的純文字 AIMessage
4. 重新插入帶有 `tool_calls` 的 AIMessage 及 `ToolMessage`，確保訊息歷史格式一致

In [6]:
_TEXT_TOOL_RE = re.compile(r"call:(\w+)\{([^}]*)\}")

def _parse_text_tool_call(content: str) -> tuple[str | None, dict]:
    m = _TEXT_TOOL_RE.search(content)
    if not m:
        return None, {}
    name = m.group(1)
    args: dict = {}
    for kv in m.group(2).split(","):
        if ":" in kv:
            k, v = kv.split(":", 1)
            args[k.strip()] = v.strip()
    return name, args

# 快速測試 parser
test_cases = [
    "call:list_files{pattern:*.md}",
    "讓我搜尋一下 call:grep{pattern:晨星科技,max_results:5} 的結果",
    "這是一般回覆，不含工具呼叫",
]
for tc in test_cases:
    name, args = _parse_text_tool_call(tc)
    print(f"Input: {tc!r}")
    print(f"  → name={name!r}, args={args}\n")

Input: 'call:list_files{pattern:*.md}'
  → name='list_files', args={'pattern': '*.md'}

Input: '讓我搜尋一下 call:grep{pattern:晨星科技,max_results:5} 的結果'
  → name='grep', args={'pattern': '晨星科技', 'max_results': '5'}

Input: '這是一般回覆，不含工具呼叫'
  → name=None, args={}



## 4 · Graph Nodes

定義三個節點函式：
- **`agent_node`**：將系統提示加到訊息最前端，呼叫綁定工具的 LLM
- **`route`**：檢查最後一則訊息，決定路由走向
- **`text_tool_node`**：處理 Gemma-4 的文字格式工具呼叫

In [7]:
def agent_node(state: MessagesState) -> dict:
    msgs = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(msgs)]}

def route(state: MessagesState) -> Literal["tools", "text_tools", "__end__"]:
    last = state["messages"][-1]
    if not isinstance(last, AIMessage):
        return END
    if last.tool_calls:
        return "tools"
    if isinstance(last.content, str) and _TEXT_TOOL_RE.search(last.content):
        return "text_tools"
    return END

In [8]:
def text_tool_node(state: MessagesState) -> dict:
    last = state["messages"][-1]
    name, args = _parse_text_tool_call(str(last.content))
    if not name:
        return {"messages": []}
    tool = _tools_by_name.get(name)
    try:
        result = tool.invoke(args) if tool else f"Unknown tool: {name}"
    except Exception as exc:
        result = f"Error calling {name}: {exc}"
    call_id = uuid.uuid4().hex[:8]
    # RemoveMessage 移除原本的純文字 AIMessage，
    # 再插入格式正確的 AIMessage（含 tool_calls）與 ToolMessage，
    # 確保訊息歷史對後續 LLM 呼叫保持一致的結構。
    return {"messages": [
        RemoveMessage(id=last.id),
        AIMessage(
            content="",
            tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}],
        ),
        ToolMessage(content=str(result), tool_call_id=call_id),
    ]}

## 5 · Build Graph

將各節點與邊組裝成 LangGraph `StateGraph`，並編譯為可執行的 agent。

In [9]:
_builder = StateGraph(MessagesState)
_builder.add_node("agent", agent_node)
_builder.add_node("tools", ToolNode(tools))
_builder.add_node("text_tools", text_tool_node)
_builder.add_edge(START, "agent")
_builder.add_conditional_edges("agent", route)
_builder.add_edge("tools", "agent")
_builder.add_edge("text_tools", "agent")
agent = _builder.compile()
print("Graph compiled ✓")

Graph compiled ✓


## 6 · 問答示範

以下五題均需要多步工具呼叫才能回答。每個 cell 展示完整的推理過程：

- **Step N: tool(args)** — agent 決定呼叫的工具與參數
- 工具回傳結果（前 500 字）
- **A: 最終答案**

這讓你清楚看到 ReAct 的「思考 → 行動 → 觀察」循環如何運作。


In [10]:
QUESTIONS = [
    "晨星科技最初決定採用什麼部署方案？後來為何改變？是誰提出異議、理由是什麼？",
    "陳建宏目前同時負責哪些專案？跨專案的人力衝突帶來哪些具體風險？",
    "新星金融科技的 PoC 通過條件是什麼？如果未通過，合作會如何發展？",
    "哪個客戶面臨最緊迫的法遵稽核壓力？稽核截止日期與具體要求為何？",
    "請列出目前所有客戶專案的預定 Go-Live 日期，並指出哪個時程的風險最高及原因。",
]

In [11]:
def stream_trace(question: str, app) -> None:
    """Stream the ReAct trace: print each tool call + result as it completes."""
    print(f"Q: {question}\n")
    step = 0

    for chunk in app.stream(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 30},
        stream_mode="updates",
    ):
        for _node, state_update in chunk.items():
            for msg in state_update.get("messages", []):
                if isinstance(msg, RemoveMessage):
                    continue
                if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        step += 1
                        args_str = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
                        header = f"─── Step {step}: {tc['name']}({args_str}) "
                        print(header + "─" * max(2, 55 - len(header)), flush=True)
                elif isinstance(msg, ToolMessage):
                    content = str(msg.content)
                    preview = content[:500]
                    print("  " + preview.replace("\n", "\n  "), flush=True)
                    if len(content) > 500:
                        print("  ...", flush=True)
                    print(flush=True)
                elif isinstance(msg, AIMessage) and not getattr(msg, "tool_calls", None):
                    if isinstance(msg.content, str) and msg.content.strip():
                        print("─" * 55)
                        print(f"A: {msg.content}", flush=True)


In [12]:
# Q1: 跨檔案決策追蹤
QUESTION = "晨星科技最初決定採用什麼部署方案？後來為何改變？是誰提出異議、理由是什麼？"
stream_trace(QUESTION, agent)


Q: 晨星科技最初決定採用什麼部署方案？後來為何改變？是誰提出異議、理由是什麼？

───────────────────────────────────────────────────────
A: call:list_files{pattern:*.md}
─── Step 1: list_files(pattern='*.md') ────────────────
  ['meeting_001_晨星科技_2026-05-08.md', 'meeting_002_鴻圖零售_2026-05-15.md', 'meeting_003_晨星科技_2026-05-22.md', 'meeting_004_新星金融科技_2026-05-28.md']

───────────────────────────────────────────────────────
A: call:grep{pattern:部署方案}
─── Step 2: grep(pattern='部署方案') ──────────────────────
  ['meeting_003_晨星科技_2026-05-22.md:32: 晨星科技 ERP 升級 — 技術規格確認暨部署方案變更討論', 'meeting_003_晨星科技_2026-05-22.md:63: | R-003 | **[升級為高]** 部署方案變更導致工期重算，原定 2026-07-01 上線風險極高 | 高 | 高 | 立即重新評估工期，考慮分階段或延期 |']

───────────────────────────────────────────────────────
A: call:read_file{path:meeting_003_晨星科技_2026-05-22.md}
─── Step 3: read_file(path='meeting_003_晨星科技_2026-05-22.md') ──
  # 客戶會議紀錄
  
  ## 基本資訊
  
  | 欄位 | 內容 |
  |------|------|
  | **客戶名稱** | 晨星科技股份有限公司 |
  | **會議日期** | 2026-05-22 |
  | **會議時間** | 15:00 – 17:30 |
  | **會議地點** | 晨

In [13]:
# Q2: 人員跨專案負載分析
QUESTION = "陳建宏目前同時負責哪些專案？跨專案的人力衝突帶來哪些具體風險？"
stream_trace(QUESTION, agent)


Q: 陳建宏目前同時負責哪些專案？跨專案的人力衝突帶來哪些具體風險？

───────────────────────────────────────────────────────
A: call:grep{pattern:陳建宏}
─── Step 1: grep(pattern='陳建宏') ───────────────────────
  ['meeting_001_晨星科技_2026-05-08.md:26: | 陳建宏 | 技術負責人（Tech Lead） | 技術評估 |', 'meeting_001_晨星科技_2026-05-08.md:76: | D-004 | 技術規格書由我方（陳建宏）撰寫，客戶方（張志偉）確認 | 王雅婷 | 2026-05-08 |', 'meeting_001_晨星科技_2026-05-08.md:85: 林佳慧           陳建宏              張志偉          陳建宏', 'meeting_001_晨星科技_2026-05-08.md:91: 陳建宏           張志偉        王雅婷          王雅婷', 'meeting_001_晨星科技_2026-05-08.md:100: | 技術規格書撰寫 | 陳建宏 | 2026-05-20 | 技術設計文件 |', 'meeting_001_晨星科技_2026-05-08.md:102: | 雲端環境建置 | 陳建宏 | 2026-06-01 | 環境建置報告 |', 'meeting
  ...

───────────────────────────────────────────────────────
A: 根據會議紀錄，陳建宏目前同時負責以下兩個專案：

1.  **晨星科技 ERP 專案**：擔任技術負責人（Tech Lead）。
2.  **新星金融科技專案**：擔任技術負責人（Tech Lead）。

**跨專案人力衝突帶來的具體風險如下：**

*   **時間衝突風險**：新星金融科技專案預計於 2026-07-01 啟動，此時間點恐與晨星科技專案 Go-Live 後的支援期重疊，導致人力投入不足（來源：`meeting_003_晨星科技_2026-05-22.md`、`meeting_004_新星金

In [14]:
# Q3: 合約條件與替代方案推理
QUESTION = "新星金融科技的 PoC 通過條件是什麼？如果未通過，合作會如何發展？"
stream_trace(QUESTION, agent)


Q: 新星金融科技的 PoC 通過條件是什麼？如果未通過，合作會如何發展？

───────────────────────────────────────────────────────
A: call:list_files{pattern:*.md}
─── Step 1: list_files(pattern='*.md') ────────────────
  ['meeting_001_晨星科技_2026-05-08.md', 'meeting_002_鴻圖零售_2026-05-15.md', 'meeting_003_晨星科技_2026-05-22.md', 'meeting_004_新星金融科技_2026-05-28.md']

───────────────────────────────────────────────────────
A: call:grep{pattern:PoC}
─── Step 2: grep(pattern='PoC') ───────────────────────
  ['meeting_004_新星金融科技_2026-05-28.md:61: | R-001 | **[高]** 10,000 TPS + 50ms 延遲要求，技術難度高，需 PoC 驗證 | 高 | 中 | 先進行 2 週 PoC，確認架構可行性後再簽約 |', 'meeting_004_新星金融科技_2026-05-28.md:74: | D-002 | 先執行 2 週技術 PoC（效能驗證），通過後正式簽約 | 方怡君、陳建宏 | 2026-05-28 |', 'meeting_004_新星金融科技_2026-05-28.md:75: | D-003 | **正式開發開始時間：2026-07-01**（依賴 PoC 通過與合約簽署） | 方怡君 | 2026-05-28 |', 'meeting_004_新星金融科技_2026-05-28.md:84: [PoC 執行] ──► [PoC 成果評估] ──► [合約簽署] ──► [架構設計] ──► [正式開發]', 'meeting_004_新星金融科技_2026-05-28.md:107: | 技術 PoC（效能驗證） | 
  ...

──────────────────────────

In [15]:
# Q4: 法遵風險與截止日期識別
QUESTION = "哪個客戶面臨最緊迫的法遵稽核壓力？稽核截止日期與具體要求為何？"
stream_trace(QUESTION, agent)


Q: 哪個客戶面臨最緊迫的法遵稽核壓力？稽核截止日期與具體要求為何？

───────────────────────────────────────────────────────
A: call:list_files{pattern:*.md}
─── Step 1: list_files(pattern='*.md') ────────────────
  ['meeting_001_晨星科技_2026-05-08.md', 'meeting_002_鴻圖零售_2026-05-15.md', 'meeting_003_晨星科技_2026-05-22.md', 'meeting_004_新星金融科技_2026-05-28.md']

───────────────────────────────────────────────────────
A: call:grep{pattern:稽核|法遵|截止日期}
─── Step 2: grep(pattern='稽核|法遵|截止日期') ────────────────
  ['meeting_001_晨星科技_2026-05-08.md:41: 3. 新系統需支援角色型存取控制（RBAC），並通過 ISO 27001 稽核要求。', 'meeting_001_晨星科技_2026-05-08.md:50: - **痛點 2｜稽核追蹤困難**：現有系統缺乏完整 Audit Log，法遵壓力大。', 'meeting_004_新星金融科技_2026-05-28.md:41: 3. 支援規則動態調整（無需重新部署），法遵團隊可自助設定風控規則。', 'meeting_004_新星金融科技_2026-05-28.md:42: 4. 提供完整稽核軌跡（Audit Trail），符合金管會《電子支付機構資訊安全標準》。', 'meeting_004_新星金融科技_2026-05-28.md:52: - **痛點 3｜法遵合規缺口**：金管會 2026 Q3 稽核在即，現有系統缺少完整 Audit Trail，風險極高。', 'meeting_004_新星金融科技_2026-05-28.md:62: | R-002 | **[高]** 金管會 Q3 稽核（預計 2026
  ...

──────────────────────

In [16]:
# Q5: 跨客戶時程彙整與風險評估
QUESTION = "請列出目前所有客戶專案的預定 Go-Live 日期，並指出哪個時程的風險最高及原因。"
stream_trace(QUESTION, agent)


Q: 請列出目前所有客戶專案的預定 Go-Live 日期，並指出哪個時程的風險最高及原因。

───────────────────────────────────────────────────────
A: call:list_files{pattern:*.md}
─── Step 1: list_files(pattern='*.md') ────────────────
  ['meeting_001_晨星科技_2026-05-08.md', 'meeting_002_鴻圖零售_2026-05-15.md', 'meeting_003_晨星科技_2026-05-22.md', 'meeting_004_新星金融科技_2026-05-28.md']

───────────────────────────────────────────────────────
A: call:grep{pattern:Go-Live}
─── Step 2: grep(pattern='Go-Live') ───────────────────
  ['meeting_001_晨星科技_2026-05-08.md:75: | D-003 | 目標上線日期：**2026-07-01**（財務模組 Go-Live） | 李明哲 | 2026-05-08 |', 'meeting_001_晨星科技_2026-05-08.md:89: [資料遷移沙盒測試] ──► [UAT] ──► [上線準備] ──► [Go-Live]', 'meeting_001_晨星科技_2026-05-08.md:106: | 正式 Go-Live | 王雅婷 | 2026-07-01 | 上線確認報告 |', 'meeting_002_鴻圖零售_2026-05-15.md:88: [整合測試（SIT）] ──► [UAT] ──► [上線準備] ──► [Go-Live]', 'meeting_002_鴻圖零售_2026-05-15.md:105: | 正式 Go-Live | 王雅婷 | 2026-07-15 | 上線確認報告 |', 'meeting_003_晨星科技_2026-05-22.md:86: > 注意：以下時程基於 2026-06-15 Go
  ...

──────────────

## 7 · 強制工具順序的 Graph 設計

### 問題

原版 `route()` 只要 `tool_calls` 為空就進入 `END`，LLM 可能只憑 grep 的片段結果就直接給答案，跳過了 `read_file`。

### 解法：守衛節點（Guard Node）

```
START → [agent] → route_v2() → [tools]      → [agent] → ...
                             → [text_tools]  → [agent] → ...
                             → [must_read]   → [agent] → (被要求呼叫 read_file)
                             → END           ← 僅在 read_file 被呼叫後才放行
```

`route_v2()` 在 agent 準備輸出最終答案前，先掃描歷史訊息是否存在 `read_file` 的工具呼叫紀錄。若沒有，路由到 `must_read` 節點注入一條提示，強迫 agent 折返並執行 `read_file`。


In [17]:
from langchain_core.messages import HumanMessage


def _was_read_file_called(messages: list) -> bool:
    """Return True if read_file was invoked at any point in the message history."""
    return any(
        tc.get("name") == "read_file"
        for msg in messages
        if isinstance(msg, AIMessage)
        for tc in getattr(msg, "tool_calls", [])
    )


def route_v2(state: MessagesState) -> Literal["tools", "text_tools", "must_read", "__end__"]:
    last = state["messages"][-1]
    if not isinstance(last, AIMessage):
        return END
    if last.tool_calls:
        return "tools"
    if isinstance(last.content, str) and _TEXT_TOOL_RE.search(last.content):
        return "text_tools"
    # Agent is about to give a final answer — only allow if read_file was used.
    if not _was_read_file_called(state["messages"]):
        return "must_read"
    return END


def must_read_node(state: MessagesState) -> dict:
    """Guard: agent tried to answer without reading any file. Redirect it."""
    return {"messages": [
        HumanMessage(content=(
            "你尚未使用 read_file 讀取任何完整文件，無法確保答案正確性。"
            "請先呼叫 read_file 閱讀相關會議紀錄的完整內容，再提供最終答案。"
        ))
    ]}


In [18]:
_builder_v2 = StateGraph(MessagesState)
_builder_v2.add_node("agent", agent_node)
_builder_v2.add_node("tools", ToolNode(tools))
_builder_v2.add_node("text_tools", text_tool_node)
_builder_v2.add_node("must_read", must_read_node)
_builder_v2.add_edge(START, "agent")
_builder_v2.add_conditional_edges("agent", route_v2)
_builder_v2.add_edge("tools", "agent")
_builder_v2.add_edge("text_tools", "agent")
_builder_v2.add_edge("must_read", "agent")
agent_v2 = _builder_v2.compile()
print("agent_v2 compiled ✓")


agent_v2 compiled ✓


### 比較：同一題，兩個 agent 的工具呼叫序列

選用 Q2（陳建宏跨專案人力衝突）示範 — 此類問題模型可能只靠 `grep` 就給答案，而不去 `read_file` 讀完整原文，導致遺漏細節。


In [19]:
QUESTION = QUESTIONS[1]  # Q2: 陳建宏跨專案人力衝突

print("═" * 55)
print("▶ Agent v1 — 原版（無強制）")
print("═" * 55)
stream_trace(QUESTION, agent)

print("\n\n" + "═" * 55)
print("▶ Agent v2 — 強制 read_file")
print("═" * 55)
stream_trace(QUESTION, agent_v2)


═══════════════════════════════════════════════════════
▶ Agent v1 — 原版（無強制）
═══════════════════════════════════════════════════════
Q: 陳建宏目前同時負責哪些專案？跨專案的人力衝突帶來哪些具體風險？

───────────────────────────────────────────────────────
A: call:grep{pattern:陳建宏}
─── Step 1: grep(pattern='陳建宏') ───────────────────────
  ['meeting_001_晨星科技_2026-05-08.md:26: | 陳建宏 | 技術負責人（Tech Lead） | 技術評估 |', 'meeting_001_晨星科技_2026-05-08.md:76: | D-004 | 技術規格書由我方（陳建宏）撰寫，客戶方（張志偉）確認 | 王雅婷 | 2026-05-08 |', 'meeting_001_晨星科技_2026-05-08.md:85: 林佳慧           陳建宏              張志偉          陳建宏', 'meeting_001_晨星科技_2026-05-08.md:91: 陳建宏           張志偉        王雅婷          王雅婷', 'meeting_001_晨星科技_2026-05-08.md:100: | 技術規格書撰寫 | 陳建宏 | 2026-05-20 | 技術設計文件 |', 'meeting_001_晨星科技_2026-05-08.md:102: | 雲端環境建置 | 陳建宏 | 2026-06-01 | 環境建置報告 |', 'meeting
  ...

───────────────────────────────────────────────────────
A: 根據會議紀錄，陳建宏目前同時負責以下兩個專案：

1.  **晨星科技 ERP 專案**：擔任技術負責人（Tech Lead）。
2.  **新星金融科技專案**：擔任技術負責人（Tech Lead）。

**跨專案人力衝突帶來的具體風險如下：**